# Sabaic OCR — Real Pilot (5 Images)

هذا الـNotebook مخصص لاختبار النموذج `synthetic_v2/best.pt` على **5 صور حقيقية تجريبية** قبل أي Fine-tuning.

الهدف:
- تجهيز جلسة Colab جديدة من الصفر.
- ربط checkpoint المحفوظ في Google Drive.
- رفع/استرجاع Dataset الخمس صور بعد تصحيح Class IDs.
- فحص الـLabels.
- قياس الـDomain Gap بدون تدريب.
- تشغيل inference عند `confidence=0.80` وحفظ الصور والنتائج في Drive.

> مهم: هذه المرحلة **اختبار فقط**. لا يتم تدريب النموذج على الصور الخمس.


## 0) تجهيز المستودع والبيئة

شغّل هذه الخلية أولًا. وجود GPU مفضل لكنه غير إلزامي لاختبار 5 صور فقط.


In [ ]:
import os, subprocess, torch
from pathlib import Path

assert os.path.exists('/content'), 'This notebook is intended for Google Colab.'

repo = Path('/content/Sabaic-OCR-YOLO')
if not repo.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/7eaur/Sabaic-OCR-YOLO.git', str(repo)],
        check=True
    )
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)

os.chdir(repo)
subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
subprocess.run(
    ['pip', 'install', '-e', '.', '--no-deps', '--no-build-isolation', '-q'],
    check=True
)
subprocess.run(['pytest', '-q'], check=True)

print('repo:', repo)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda_device:', torch.cuda.get_device_name(0))
else:
    print('device note: CPU is acceptable for this 5-image pilot.')


## 1) ربط Google Drive واسترجاع النموذج v2

يجب أن يكون الملف التالي موجودًا في Drive:

`MyDrive/Sabaic-OCR-YOLO/checkpoints/synthetic_v2/best.pt`


In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Sabaic-OCR-YOLO')
DRIVE_CKPT = DRIVE_ROOT / 'checkpoints'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)

os.chdir('/content/Sabaic-OCR-YOLO')
local_ckpt = Path('checkpoints')

if local_ckpt.is_symlink():
    local_ckpt.unlink()
elif local_ckpt.exists():
    shutil.rmtree(local_ckpt)

local_ckpt.symlink_to(DRIVE_CKPT, target_is_directory=True)

BEST = Path('checkpoints/synthetic_v2/best.pt')
print('v2 best exists:', BEST.exists())
print('checkpoint path:', BEST.resolve() if BEST.exists() else BEST)

assert BEST.exists(), (
    'synthetic_v2/best.pt غير موجود في Google Drive. '
    'لا تكمل قبل استرجاع checkpoint الصحيح.'
)


## 2) استرجاع أو رفع الخمس صور المصححة

في أول تشغيل سيطلب منك رفع:

`222_fixed_project_ids.zip`

بعدها سيتم حفظ نسخة جاهزة داخل Google Drive، لذلك في أي جلسة لاحقة سيتم استرجاعها تلقائيًا بدون رفعها مرة أخرى.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

os.chdir('/content/Sabaic-OCR-YOLO')

LOCAL_PILOT = Path('data/real_pilot_5')
LOCAL_IMAGES = LOCAL_PILOT / 'images'
LOCAL_LABELS = LOCAL_PILOT / 'labels'

DRIVE_PILOT = DRIVE_ROOT / 'real_data/pilot_5'
DRIVE_IMAGES = DRIVE_PILOT / 'images'
DRIVE_LABELS = DRIVE_PILOT / 'labels'

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def image_count(path):
    return len([p for p in path.glob('*') if p.suffix.lower() in IMAGE_EXTS]) if path.exists() else 0

def label_count(path):
    return len(list(path.glob('*.txt'))) if path.exists() else 0

if image_count(DRIVE_IMAGES) == 5 and label_count(DRIVE_LABELS) == 5:
    print('Restoring prepared real pilot from Google Drive...')
    if LOCAL_PILOT.exists():
        shutil.rmtree(LOCAL_PILOT)
    shutil.copytree(DRIVE_PILOT, LOCAL_PILOT)
else:
    print('No prepared pilot found in Drive.')
    print('Upload: 222_fixed_project_ids.zip')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise RuntimeError('لم يتم رفع ملف ZIP.')

    zip_name = zip_names[0]
    extract_root = Path('/content/real_pilot_upload')
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)

    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall(extract_root)

    all_images = [
        p for p in extract_root.rglob('*')
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]
    all_labels = [
        p for p in extract_root.rglob('*.txt')
        if p.is_file()
    ]

    label_by_stem = {p.stem: p for p in all_labels}
    paired = [(img, label_by_stem.get(img.stem)) for img in all_images]
    paired = [(img, lab) for img, lab in paired if lab is not None]

    if len(paired) != 5:
        raise RuntimeError(
            f'Expected 5 image/label pairs, found {len(paired)}. '
            'تأكد أنك رفعت النسخة المصححة.'
        )

    if LOCAL_PILOT.exists():
        shutil.rmtree(LOCAL_PILOT)
    LOCAL_IMAGES.mkdir(parents=True)
    LOCAL_LABELS.mkdir(parents=True)

    for img, lab in sorted(paired, key=lambda x: x[0].name):
        shutil.copy2(img, LOCAL_IMAGES / img.name)
        shutil.copy2(lab, LOCAL_LABELS / f'{img.stem}.txt')

    if DRIVE_PILOT.exists():
        shutil.rmtree(DRIVE_PILOT)
    DRIVE_PILOT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(LOCAL_PILOT, DRIVE_PILOT)
    print('Prepared pilot saved to Google Drive.')

print('images:', image_count(LOCAL_IMAGES))
print('labels:', label_count(LOCAL_LABELS))
print('local pilot:', LOCAL_PILOT.resolve())
print('drive pilot:', DRIVE_PILOT)


## 3) فحص الـLabels قبل الاختبار

هذه الخلية تتحقق من:
- وجود 5 صور و5 ملفات labels.
- كل Class ID داخل نطاق مشروعنا `0..29`.
- إحداثيات YOLO صحيحة.
- عدد الـBounding Boxes.


In [ ]:
from pathlib import Path
from collections import Counter
from sabaic_ocr.data.dataset import read_yolo_labels, list_images

images = list_images(LOCAL_IMAGES)
assert len(images) == 5, f'Expected 5 images, found {len(images)}'

class_hist = Counter()
total_boxes = 0
missing = []

for img in images:
    lab = LOCAL_LABELS / f'{img.stem}.txt'
    if not lab.exists():
        missing.append(img.name)
        continue
    targets = read_yolo_labels(lab, num_classes=30)
    total_boxes += len(targets)
    class_hist.update(int(x) for x in targets[:, 0].tolist())

assert not missing, f'Missing labels for: {missing}'

print('images:', len(images))
print('total_boxes:', total_boxes)
print('classes_present:', sorted(class_hist))
print('class_histogram:', dict(sorted(class_hist.items())))
print('LABEL VALIDATION: PASS')


## 4) قياس الـDomain Gap بدون تدريب

نختبر النموذج الحالي على الصور الخمس عند عدة thresholds:
- `0.25`
- `0.50`
- `0.80`

النتيجة الأهم هنا:
- Localization precision / recall.
- Classification accuracy على الـboxes التي تم تحديد مكانها بشكل صحيح.
- Same-class recall.

هذا الاختبار لا يغير أوزان النموذج.


In [ ]:
import subprocess, os
from pathlib import Path

os.chdir('/content/Sabaic-OCR-YOLO')

RESULTS_DIR = DRIVE_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DIAG_OUT = RESULTS_DIR / 'real_pilot_5_diagnostic.json'

subprocess.run([
    'python', 'scripts/diagnose_checkpoint.py',
    '--checkpoint', 'checkpoints/synthetic_v2/best.pt',
    '--images', str(LOCAL_IMAGES),
    '--labels', str(LOCAL_LABELS),
    '--batch-size', '5',
    '--iou', '0.45',
    '--conf', '0.25', '0.50', '0.80',
    '--output', str(DIAG_OUT),
], check=True)

print('Saved diagnostic:', DIAG_OUT)


## 5) ملخص واضح لنتيجة الاختبار

شغّل هذه الخلية بعد انتهاء التشخيص وأرسل لي الناتج منها.


In [ ]:
import json
from pathlib import Path

report = json.loads(DIAG_OUT.read_text(encoding='utf-8'))

print('checkpoint:', report['checkpoint'])
print('test_images:', report['test_images'])
print()

for r in report['threshold_runs']:
    print(
        f"conf={r['confidence_threshold']:.2f} "
        f"pred={r['predictions']} "
        f"gt={r['ground_truth_boxes']} "
        f"loc_precision={r['localization_precision_iou50']:.4f} "
        f"loc_recall={r['localization_recall_iou50']:.4f} "
        f"cls_acc={r['classification_accuracy_on_localized_matches']:.4f} "
        f"same_class_recall={r['end_to_end_same_class_recall_iou50']:.4f} "
        f"mean_iou={r['mean_iou_of_localized_matches']:.4f}"
    )


## 6) تشغيل Inference بصري عند confidence = 0.80

بعد مراجعة نتائج القسم 5، يمكن تشغيل هذه الخلية لرؤية الـBounding Boxes التي يتنبأ بها النموذج على الصور الحقيقية.

النتائج تحفظ في Google Drive.


In [ ]:
import subprocess, os
from pathlib import Path

os.chdir('/content/Sabaic-OCR-YOLO')
OUT_DIR = DRIVE_ROOT / 'outputs/real_pilot_5/conf_080'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for img in sorted(LOCAL_IMAGES.iterdir()):
    if img.suffix.lower() not in IMAGE_EXTS:
        continue
    print('\n===', img.name, '===')
    subprocess.run([
        'python', 'scripts/infer.py',
        '--checkpoint', 'checkpoints/synthetic_v2/best.pt',
        '--image', str(img),
        '--conf', '0.80',
        '--iou', '0.45',
        '--output-dir', str(OUT_DIR),
    ], check=True)

print('\nSaved inference outputs:', OUT_DIR)


## 7) عرض الصور الخمس بعد التنبؤ

تعرض هذه الخلية الصور الناتجة التي تحتوي على Bounding Boxes المتنبأ بها.


In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

box_images = sorted(OUT_DIR.glob('*_boxes.jpg'))
print('annotated images:', len(box_images))

for p in box_images:
    print('\n', p.name)
    display(Image.open(p))


## القرار بعد هذه التجربة

بعد إرسال ناتج القسم **5**:
- إذا كان localization جيدًا والتصنيف ضعيفًا → نثبت أن هناك Domain Gap في شكل الحروف.
- إذا كان localization والتصنيف كلاهما ضعيفًا → نحتاج Real Fine-tuning بشكل أوضح.
- لا يتم تدريب النموذج على هذه الخمس صور قبل توثيق نتيجة الاختبار الأولي.

الـFine-tuning الرسمي لاحقًا يحتاج **200+ صورة حقيقية labeled للتدريب** مع Validation/Test حقيقية منفصلة.
